# Crate Download Analysis

This notebook pulls version metadata and download stats for `CRATE` from crates.io, then summarizes overall activity, recent version usage, and a simple trend visualization.

In [5]:
import requests
import pandas as pd

CRATE = "jupyter-protocol"
HEADERS = {"User-Agent": f"{CRATE}-analysis"}

# Fetch all version metadata
crate_resp = requests.get(f"https://crates.io/api/v1/crates/{CRATE}", headers=HEADERS)
crate_data = crate_resp.json()

# Build version lookup: id -> version number
versions = {v["id"]: v["num"] for v in crate_data["versions"]}

# All versions with total downloads
versions_df = pd.DataFrame(
    [
        {
            "version": v["num"],
            "downloads": v["downloads"],
            "published": v["created_at"][:10],
        }
        for v in crate_data["versions"]
    ]
).sort_values("downloads", ascending=False)

print(f"Total crate downloads: {crate_data['crate']['downloads']:,}")
print(f"Total versions: {len(versions_df)}\n")
versions_df

Total crate downloads: 205,907
Total versions: 21


,version,downloads,published
13,0.8.0,70101,2025-06-18
15,0.6.0,42080,2025-01-09
10,0.10.0,30393,2025-11-13
3,1.2.1,17898,2026-02-20
16,0.5.0,15568,2024-12-02
4,1.2.0,4757,2026-02-11
18,0.3.0,4333,2024-11-23
0,1.4.0,3168,2026-02-26
19,0.2.0,2980,2024-11-14
8,0.11.0,2700,2025-12-17


## Recent Download Activity

Fetch daily download counts from crates.io, map version IDs to version strings, and summarize the last 30 days of activity.

In [6]:
# Fetch daily download data (last 90 days) with version ID mapping
downloads_resp = requests.get(
    f"https://crates.io/api/v1/crates/{CRATE}/downloads", headers=HEADERS
)
downloads_data = downloads_resp.json()

# Map version IDs to version names
daily = pd.DataFrame(downloads_data["version_downloads"])
daily["version"] = daily["version"].map(versions)
daily["date"] = pd.to_datetime(daily["date"])

# Pivot: dates as rows, versions as columns, downloads as values
pivot = daily.pivot_table(
    index="date", columns="version", values="downloads", fill_value=0
)

# Exclude the latest day so a partial bucket does not distort the tail.
pivot = pivot[pivot.index < pivot.index.max()]

# Focus on last 30 complete days
last_30 = pivot[pivot.index >= pivot.index.max() - pd.Timedelta(days=30)]

# Show totals per version in the last 30 days, sorted
totals = last_30.sum().sort_values(ascending=False)
print("Downloads by version (last 30 complete days):\n")
for ver, count in totals.items():
    if count > 0:
        print(f"  {ver:>8s}: {count:,}")
print(f"\n  {'TOTAL':>8s}: {totals.sum():,}")

Downloads by version (last 30 complete days):

     1.2.1: 17,343.0
     1.2.0: 4,844.0
     1.4.0: 3,062.0
     1.3.0: 71.0
     1.2.2: 17.0

     TOTAL: 25,337.0


## Visualization

Plot overall daily downloads for the crate and a stacked breakdown of the versions that saw activity in the last 30 days.

In [7]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.style.use("dark_background")

# Daily total downloads across all versions
daily_total = pivot.sum(axis=1)

# Top versions by recent downloads for breakdown
top_versions = totals[totals > 0].index.tolist()
recent_pivot = pivot[top_versions]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
fig.patch.set_alpha(0)
for ax in (ax1, ax2):
    ax.set_facecolor("none")
    ax.spines["bottom"].set_color("#555")
    ax.spines["left"].set_color("#555")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(colors="#aaa")
    ax.yaxis.label.set_color("#ccc")
    ax.title.set_color("#ddd")

# Top: total daily downloads
ax1.fill_between(daily_total.index, daily_total.values, alpha=0.3, color="#569cd6")
ax1.plot(daily_total.index, daily_total.values, linewidth=1.5, color="#569cd6")
ax1.set_ylabel("Downloads")
ax1.set_title(f"{CRATE} — Daily Downloads (last 90 days)")
ax1.grid(True, alpha=0.3, color="#888")

# Bottom: stacked area by version
ax2.stackplot(
    recent_pivot.index,
    *[recent_pivot[v] for v in top_versions],
    labels=top_versions,
    alpha=0.85,
)
ax2.set_ylabel("Downloads")
ax2.set_title("Downloads by Version")
ax2.legend(
    loc="upper left",
    fontsize=9,
    facecolor="#1e1e1e",
    edgecolor="#555",
    labelcolor="#ccc",
)
ax2.grid(True, alpha=0.3, color="#888")
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax2.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))

output_path = f"/tmp/{CRATE}.png"
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(output_path, transparent=True, dpi=150)
plt.show()